# *<center> The Ion Funnel: moving ions through a gas </center>*

### The following notebook adds a GAS. Notebooks 01 and 02 flew ions through vacuum; here the ion collides with a neutral thousands of times on its way through the device, and that changes what the instrument is for. Specific topics include:

* The funnel geometry: a stack of rings whose apertures shrink, driven by
  RF on alternating rings and a DC ladder along the stack.
* Why the RF confines rather than ejects — the pseudopotential wall, seen
  in the solved field.
* **Integration time versus record time**: `dt_ns` sets the physics,
  `rec_every` sets only what is stored, and confusing the two is a
  classic way to draw a wrong conclusion from a right simulation.
* **Kinetic energy as a function of distance through the funnel** — the
  plot that shows what the gas is doing.
* The energy budget: how much of the field's work reaches the ion, and
  how much is handed to the gas.

### Conventions used in this document:

* **Units are mm, eV, microseconds, volts and Torr** unless a name says
  otherwise; every parameter carries its unit in its name or a comment.
* **CAPITALS are parameters you are meant to change**; they sit in a cell
  immediately before the stage that first uses them.
* **The axial coordinate is `x`** on this r-z device (the field was solved
  in one half-plane and revolved); `y` is the radial coordinate, so
  $r = |y|$ on axis-crossing trajectories.
* **`spec` is the declaration, `model` is the solved field, `fly` is the
  integrator.** Anything read off `model` is what the kernel actually
  flew — display equals solve.
* Equations are numbered (**Eqn #1**, **Eqn #2**, ...) and the code that
  implements one names it.

#### Relevant References

* A novel ion funnel for focusing ions at elevated pressure using
  electrospray ionization mass spectrometry
    * S. A. Shaffer, K. Tang, G. A. Anderson, D. C. Prior, H. R. Udseth
      and R. D. Smith
        * *Rapid Commun. Mass Spectrom.* **11**, 1813-1817 (1997)
        * doi.org/10.1002/(SICI)1097-0231(19971030)11:16<1813::AID-RCM87>3.0.CO;2-D
* The ion funnel: theory, implementations, and applications
    * R. T. Kelly, A. V. Tolmachev, J. S. Page, K. Tang and R. D. Smith
        * *Mass Spectrom. Rev.* **29**, 294-312 (2010)
        * doi.org/10.1002/mas.20232
* Ion optics simulations at atmospheric pressure
    * A. D. Appelhans and D. A. Dahl
        * *Int. J. Mass Spectrom.* **244**, 1-14 (2005)
        * doi.org/10.1016/j.ijms.2005.03.010
* Radiofrequency Spectroscopy of Stored Ions I: Storage
    * H. G. Dehmelt
        * *Adv. At. Mol. Phys.* **3**, 53-72 (1967) — the pseudopotential

____

## Stage 0 — global style and constants (everything else is per-stage)

## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the tapered ring stack, the DC gradient along the axis, and example ions being squeezed toward the exit aperture in buffer gas.

Deck: `examples/ion_funnel_rz.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/ion_funnel_rz.json', banked='panel_funnel.png', height=520)


In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# path anchor: every relative path below is REPO-ROOT-relative,
# independent of where Jupyter/VS Code set the working directory.
from ion_gym.io.paths import repo_root as _repo_root
import os as _os
_os.chdir(_repo_root())

import time
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown   # imported ONCE, here

from ion_gym.io.sim_spec import SimSpec
from ion_gym.physics.sim_build import build_run, build_needs_solve
from ion_gym.physics.stats import (
    compute_stats, stats_markdown, auto_transmitted_fate,
    enabled_planes, FATE_NAME,
)
from ion_gym.viz.viz_core import (
    scene_from_simspec, describe_fates, interactive_panel,
)
from ion_gym.viz.viz_core import voxel_views_2d
from ion_gym.viz.pe_view import compute_component, pe_figure_3d
from ion_gym.physics.ensemble_driver import IonResult

# ---- FIGURE SIZE ----------------------------------------------------------
# Every panel takes a `figsize=(width, height)` keyword — matplotlib's name,
# in PIXELS (a browser's native unit). Change FIGSIZE here and re-run, or
# pass figsize= to a single call to size just that figure:
#     interactive_panel(scene, "xy", figsize=(1200, 500))
FIGSIZE = (980, 420)     # the funnel is long and thin

# Field-panel colormap: any plotly colorscale name ("Magma", "Blackbody",
# "Jet", "Hot", "Plasma", "Cividis"). Display preference only.
COLORMAP = "Magma"

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## Stage A — the device

An ion funnel is a stack of ring electrodes whose apertures shrink along
the axis. Two drives act at once, and they do different jobs:

* **RF on alternating rings** (adjacent rings in antiphase) makes a
  repulsive wall near the electrodes that pushes ions back toward the
  axis. This is the same pseudopotential as the quadrupole notebook: the
  ion is pushed away from strong RF field, and the field is strongest at
  the metal.
* **A DC ladder** — a slow, monotonic voltage drop from the entrance ring
  to the exit — is what actually moves the ion forward through the gas.
  Without it, a collision-dominated ion would simply diffuse.

The ladder is declared once as a `DCGroupSpec` and interpolated across
the member electrodes, so the profile is a *declared* property rather than
seventeen hand-typed numbers:

**Eqn #1**

$$V_i \;=\; V_{\text{in}} \;+\;
   \bigl(V_{\text{out}} - V_{\text{in}}\bigr)\,
   \frac{i}{N - 1},\qquad i = 0 \ldots N-1.$$

The cell below reads the shipped example and **measures** the aperture
taper from the electrode shapes — the geometry the solver was handed, not
a drawing of it.

In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# ---- The GEOMETRY source -- the one thing kept in the file --------------
SPEC_PATH = str(ROOT / 'examples/ion_funnel_rz.json')

spec = SimSpec.from_json(SPEC_PATH)
print(f"loaded {spec.name!r}")
for group in spec.geometry.rf_groups:
    print(f"  RF group {group.name}: {group.amplitude_v:g} V at "
          f"{group.frequency_hz/1e3:g} kHz, phase {group.phase_deg:g} deg")
for group in getattr(spec.geometry, "dc_groups", []) or []:
    print(f"  DC ladder {group.name}: {group.v_in:g} V -> {group.v_out:g} V "
          f"({group.interp})")
print(f"  gas: {spec.collisions.gas} at {spec.collisions.P_torr:.2f} Torr, "
      f"{spec.collisions.T_k:g} K, model {spec.collisions.model!r}")
print(f"  ions: m/z {spec.source.mz_list}, "
      f"{spec.source.ke_lo:g}-{spec.source.ke_hi:g} eV, "
      f"disc r = {spec.source.r_mm:g} mm")

# The aperture taper, MEASURED from the shapes the solver rasterized.
# A RING is an electrode whose metal starts at a radius above the axis
# (inner radius > 0) — that is a declared geometric property, not a guess
# from the electrode's name. An electrode whose metal reaches the axis is
# a PLATE and has no aperture to plot; it is reported separately rather
# than silently averaged into the taper (an earlier draft did exactly
# that and reported a "5.5 mm -> 0.00 mm" taper, which is nonsense).
ring_x_mm, ring_aperture_mm, ring_volts = [], [], []
plates = []
for el in spec.geometry.electrodes:
    for shape in el.shapes:
        p = shape.params
        inner_r = float(p.get("y_mm", 0.0))
        if inner_r > 0.0:
            ring_x_mm.append(float(p["x_mm"]))
            ring_aperture_mm.append(inner_r)
            ring_volts.append(float(el.dc))
        else:
            plates.append((el.name, float(p["x_mm"]), float(el.dc)))
print(f"\n{len(ring_x_mm)} rings: aperture radius "
      f"{ring_aperture_mm[0]:.2f} mm at the entrance -> "
      f"{ring_aperture_mm[-1]:.2f} mm at the last ring "
      f"(a {ring_aperture_mm[0]/ring_aperture_mm[-1]:.1f}x reduction)")
for name, x_mm, volts in plates:
    print(f"{len(plates)} solid electrode(s) reaching the axis: "
          f"{name!r} at x = {x_mm:.2f} mm, {volts:g} V — this example ends "
          "in a CLOSED plate, so 'arriving' below means landing on it; the "
          "radial spread there is what sets the conductance-limit aperture "
          "a real instrument would drill.")
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(spec, **DECK_OVERRIDES)


### The operating point: the deck of record, read back

The UI and this notebook must show the **same instrument at the same
tune**, so `examples/ion_funnel_rz.json` — the
deck the GUI loads — is the single source of truth, and every named
parameter below is **read from it**, printed with units, and consumed by
the stages. Two consequences worth noticing before any flight:

* **Ions are born inside the funnel.** The deck's source is a box
  centred at x = 5.0 mm, well inside the ring stack (first ring at
  x = 2.0 mm). Birthing upstream of the stack would start ions in
  fringe field the funnel never shaped — not good simulation practice
  for a transport study.
* **The flight is allowed to finish.** The deck's t_max covers the
  full ladder drift for every declared species (measured at seed 0:
  slowest m/z 622 exit at ~2.6 ms against t_max 3 ms); an occasional
  diffusion-unlucky heavy ion may still be flying at the clock, which
  the fate report states rather than hides.

The one deliberate divergence is the **seed**: the deck is unseeded (the
GUI announces its random draw per run), while a teaching notebook must
reproduce exactly, so the seed is pinned here and the table below marks
it. To experiment, override a value at a `load_funnel(...)` call site —
the change is then visible where it happens, not buried in this block.

In [ ]:
# ---- The operating point: READ FROM THE DECK -----------------------------
# (These are the specs the UI uses.) Every value below comes
# from examples/ion_funnel_rz.json; the names exist so the stages can
# consume and print them with units, not to introduce second copies.
_deck = SimSpec.from_json(SPEC_PATH)

# ---- ION SOURCE (deck births INSIDE the stack; first ring at 2.0 mm) ----
N_IONS      = _deck.source.n_ions        # ions per ensemble, PER m/z
MZ_LIST     = list(_deck.source.mz_list) # three species flown together
KE_LO_EV    = _deck.source.ke_lo         # birth KE window (eV)
KE_HI_EV    = _deck.source.ke_hi
TOB_SPAN_US = _deck.source.tob_span_us   # birth-time spread: RF-phase averaging
SEED        = 0   # PINNED (deck is unseeded): reproducibility is a
                  # property of a teaching notebook, not of the instrument

# ---- FLIGHT (deck values; t_max sized so every species can finish) -------
DT_NS           = _deck.integration.dt_ns
T_MAX_US        = _deck.integration.t_max_us
REC_EVERY       = _deck.integration.rec_every
RECORD_CHANNELS = list(_deck.integration.record_channels)

# ---- GAS (deck values) ---------------------------------------------------
GAS_ENABLED = _deck.collisions.enabled
GAS         = _deck.collisions.gas
P_TORR      = _deck.collisions.P_torr
T_K         = _deck.collisions.T_k

# ---- DRIVE (deck values; the table below prints them) --------------------
RF_AMPLITUDE_V  = _deck.geometry.rf_groups[0].amplitude_v
RF_FREQUENCY_HZ = _deck.geometry.rf_groups[0].frequency_hz
LADDER_V_IN     = _deck.geometry.dc_groups[0].v_in
LADDER_V_OUT    = _deck.geometry.dc_groups[0].v_out

def load_funnel(**overrides):
    """Load the deck, apply the operating point above (which IS the
    deck's, so most assignments are identities), pin the seed.

    Every stage calls this, so there is exactly one definition of "the
    funnel at its operating point". Keyword overrides let a stage change
    one thing (e.g. gas_enabled=False for the vacuum control) without
    copying the block — the change is then visible at the call site.
    The SOURCE GEOMETRY (distribution, position, extents) is NOT touched:
    it is the deck's declaration, and moving ions outside the stack is
    exactly the practice this cell exists to prevent.
    """
    sp = SimSpec.from_json(SPEC_PATH)          # geometry + source from the file

    sp.source.n_ions = int(overrides.get("n_ions", N_IONS))
    sp.source.mz_list = list(overrides.get("mz_list", MZ_LIST))
    sp.source.ke_lo = float(overrides.get("ke_lo", KE_LO_EV))
    sp.source.ke_hi = float(overrides.get("ke_hi", KE_HI_EV))
    sp.source.tob_span_us = float(TOB_SPAN_US)
    sp.source.seed = int(SEED)

    sp.integration.dt_ns = float(overrides.get("dt_ns", DT_NS))
    sp.integration.t_max_us = float(overrides.get("t_max_us", T_MAX_US))
    sp.integration.rec_every = int(overrides.get("rec_every", REC_EVERY))
    sp.integration.record_channels = list(RECORD_CHANNELS)

    sp.collisions.enabled = bool(overrides.get("gas_enabled", GAS_ENABLED))
    sp.collisions.gas = str(GAS)
    # BOTH pressure fields, atomically, through the one sanctioned door:
    # P_torr and P_pa are two fields for one quantity, so assigning P_torr
    # alone would leave the loaded deck's P_pa in place and the pair would
    # DISAGREE (validate() refuses).
    sp.collisions.set_pressure_torr(float(overrides.get("p_torr", P_TORR)))
    sp.collisions.T_k = float(T_K)

    for group in sp.geometry.rf_groups:        # both phases share amplitude
        group.amplitude_v = float(overrides.get("rf_v", RF_AMPLITUDE_V))
        group.frequency_hz = float(RF_FREQUENCY_HZ)
    for group in sp.geometry.dc_groups:
        group.v_in = float(overrides.get("v_in", LADDER_V_IN))
        group.v_out = float(overrides.get("v_out", LADDER_V_OUT))
    sp.resolve_dc_groups()   # ladder ends -> per-member dc (a RE-WEIGHT)
    return sp

# What the file says, and what this notebook flies — the table now
# demonstrates AGREEMENT; the seed row is the one declared divergence.
from_file = SimSpec.from_json(SPEC_PATH)
spec = load_funnel()
comparison = [
    ("ions", from_file.source.n_ions, spec.source.n_ions),
    ("m/z", from_file.source.mz_list, spec.source.mz_list),
    ("KE lo-hi (eV)", f"{from_file.source.ke_lo:g}-{from_file.source.ke_hi:g}",
     f"{spec.source.ke_lo:g}-{spec.source.ke_hi:g}"),
    ("source", f"{from_file.source.distribution} at "
     f"x = {from_file.source.x0_mm:g} mm",
     f"{spec.source.distribution} at x = {spec.source.x0_mm:g} mm"),
    ("dt (ns)", from_file.integration.dt_ns, spec.integration.dt_ns),
    ("t_max (us)", from_file.integration.t_max_us,
     spec.integration.t_max_us),
    ("rec_every", from_file.integration.rec_every,
     spec.integration.rec_every),
    ("gas", f"{from_file.collisions.gas} @ "
     f"{from_file.collisions.P_torr:.2f} Torr",
     f"{spec.collisions.gas} @ {spec.collisions.P_torr:.2f} Torr"),
    ("RF (V)", from_file.geometry.rf_groups[0].amplitude_v,
     spec.geometry.rf_groups[0].amplitude_v),
    ("ladder (V)", f"{from_file.geometry.dc_groups[0].v_in:g} -> "
     f"{from_file.geometry.dc_groups[0].v_out:g}",
     f"{spec.geometry.dc_groups[0].v_in:g} -> "
     f"{spec.geometry.dc_groups[0].v_out:g}"),
    ("seed", "unseeded (GUI announces its draw)", spec.source.seed),
]
lines = ["| quantity | from the JSON | flown by this notebook |", "|---|---|---|"]
lines += [f"| {a} | {b} | {c} |" for a, b, c in comparison]
display(Markdown("\n".join(lines)))

F_RF_HZ = spec.geometry.rf_groups[0].frequency_hz
T_RF_US = 1e6 / F_RF_HZ
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(from_file, **DECK_OVERRIDES)


### Declaring groups and assigning electrodes

Two group types carry every drive in the toolkit, and both are declared
objects rather than per-electrode numbers.

**`RFGroupSpec`** is a named waveform: amplitude, frequency, phase. An
electrode joins by name, and the group contributes on top of whatever DC
that electrode has:

**Eqn #2**

$$V_{\text{electrode}}(t) \;=\; V_{\text{dc}}
   \;+\; A\,\sin(2\pi f t + \varphi).$$

Two groups 180° apart is the entire trick behind both this funnel and the
quadrupole: adjacent rings (or opposite rod pairs) are driven in
antiphase.

**`DCGroupSpec`** is a ladder: `v_in` lands on the member with the lowest
`dc_index`, `v_out` on the highest, and members in between are
interpolated. Membership carries a **number** (`dc_index`), not a name to
be sorted — "ring_10" sorting before "ring_9" is exactly the kind of
silent mis-ordering that would still solve and still be wrong.

The member `dc` is **derived, not authored**: `resolve_dc_groups()`
recomputes it. And because the bases are per-electrode, sweeping a ladder
is a re-weight — a cache hit, not a re-solve.

In [ ]:
# Who belongs to what, read from the spec.
rows = ["| electrode | dc_index | dc_group | RF group | dc (V) |",
        "|---|---|---|---|---|"]
for el in spec.geometry.electrodes:
    rows.append(f"| {el.name} | {el.dc_index} | {el.dc_group or '-'} | "
                f"{', '.join(el.rf_groups) if el.rf_groups else '-'} | "
                f"{el.dc:.1f} |")
display(Markdown("\n".join(rows)))

In [ ]:
# Re-declaring the ladder: change the two ENDS, and every member follows.
for ends in ((100.0, 68.0), (100.0, 40.0)):
    trial = load_funnel(v_in=ends[0], v_out=ends[1])
    members = [e for e in trial.geometry.electrodes
               if e.dc_group == trial.geometry.dc_groups[0].name]
    volts = [f"{e.dc:.1f}" for e in members[:4]]
    axial_field = ((ends[0] - ends[1])
                   / (members[-1].shapes[0].params["x_mm"]
                      - members[0].shapes[0].params["x_mm"]))
    print(f"ladder {ends[0]:.0f} -> {ends[1]:.0f} V: first four members "
          f"{', '.join(volts)} ... "
          f"axial field {axial_field:.2f} V/mm across the stack")
print("\nThe member voltages were never typed: they are DERIVED from the "
      "two ends by resolve_dc_groups(). Changing them is a re-weight of "
      "the SAME solved bases -- the next build is a cache hit.")

In [ ]:
# Assigning an electrode to a different group: the deliberate mistake.
# Adjacent rings are in ANTIPHASE (RFA/RFB); put them all on one phase and
# the alternation that makes the pseudopotential wall disappears.
broken = load_funnel(n_ions=8)
for el in broken.geometry.electrodes:
    if el.rf_groups:                 # every RF-driven ring onto ONE group
        el.rf_groups = ["RFA"]
model_b, fly_b, cols_b, births_b = build_run(broken)
kinds_broken = [fly_b(i)[1]["kind"] for i in range(len(births_b))]

correct = load_funnel(n_ions=8)
model_c, fly_c, cols_c, births_c = build_run(correct)
kinds_ok = [fly_c(i)[1]["kind"] for i in range(len(births_c))]

def arrival_radius(fly_fn, n, cols):
    iy = cols.index("y")
    return float(np.mean([abs(fly_fn(i)[0][-1, iy]) for i in range(n)]))

r_broken = arrival_radius(fly_b, len(births_b), cols_b)
r_ok = arrival_radius(fly_c, len(births_c), cols_c)
print(f"all rings on ONE phase : mean arrival |r| = {r_broken:.3f} mm, "
      f"fates {[FATE_NAME[k] for k in set(kinds_broken)]}")
print(f"alternating phases     : mean arrival |r| = {r_ok:.3f} mm, "
      f"fates {[FATE_NAME[k] for k in set(kinds_ok)]}")
if r_broken > r_ok:
    print(f"\n-> the single-phase stack delivers ions "
          f"{r_broken/max(r_ok,1e-9):.1f}x wider. With every ring at the "
          "same potential at the same instant there is no field BETWEEN "
          "rings for most of the cycle, so the pseudopotential wall that "
          "does the focusing is largely gone.")
else:
    print(f"\n-> at this operating point the single-phase stack did NOT "
          f"deliver a wider beam ({r_broken:.3f} vs {r_ok:.3f} mm); read "
          "the fates above before concluding anything.")

### What the declaration actually builds

Before any physics, look at the device. This model is **r-z**: the solver
works one half-plane and the third dimension comes from the *declared*
rotational symmetry. `voxel_views_2d` supplies that dimension for display
— it revolves the solver's own label grid rather than re-rasterizing
anything, so what you orbit below is the metal the kernel flew through,
not a CAD drawing of it.

Three views come back, and each answers a different question:

* **`3d`** — the revolved stack: how the rings nest and taper.
* **`rz`** — the full cross-section, the solver labels mirrored about the
  axis (this is the view every 2-D figure in this notebook uses).
* **`end-on`** — exact concentric annuli looking down the axis: the
  aperture sequence an ion "sees" on its way through.

The azimuthal sampling in the `3d` view is a display choice and the title
says so; the `rz` and `end-on` views are exact.

In [ ]:
# ---- Parameters of the construction render -- change to suit your system -
CONSTRUCTION_VIEW = "3d"     # "3d" | "rz" | "end-on"

electrode_names = {i: el.name for i, el
                   in enumerate(spec.geometry.electrodes, start=1)}
electrode_volts = {i: float(el.dc) for i, el
                   in enumerate(spec.geometry.electrodes, start=1)}
model_geom, _fly_g, _cols_g, _births_g = build_run(spec)
h_mm = float(getattr(model_geom, "mm_per_gu", None) or model_geom.h_mm)

construction = voxel_views_2d(
    np.asarray(model_geom.ele), h_mm, symmetry="rz",
    names=electrode_names, voltages=electrode_volts,
    title=f"{spec.name}")
print("views returned:", ", ".join(sorted(construction)))
construction[CONSTRUCTION_VIEW]

**Answering the obvious question:** you cannot ask an r-z scene for an
`xz` or `yz` panel — `interactive_panel` refuses, because an r-z solve
has no such planes; it has a half-plane and a symmetry declaration. The
app is not doing anything different: switching views there calls exactly
the function above. Programmatically:

```python
views = voxel_views_2d(model.ele, h_mm, symmetry="rz")   # or "planar"
views["3d"]        # revolved solid
views["rz"]        # full mirrored cross-section
views["end-on"]    # concentric annuli, looking down the axis
```

For a planar (slab) route the same call returns `{"3d", "xy"}`, where the
depth of the 3-D slab is presentational and named in the title. The rule
underneath: **a view is either exact or it is labelled as a sampling of a
declared symmetry — never an invented plane.**

In [ ]:
fig_taper = go.Figure()
fig_taper.add_scatter(x=ring_x_mm, y=ring_aperture_mm, mode="lines+markers",
                      name="aperture radius")
fig_taper.add_scatter(x=ring_x_mm, y=[-a for a in ring_aperture_mm],
                      mode="lines+markers", showlegend=False,
                      line=dict(color="#1f77b4"))
fig_taper.add_scatter(x=ring_x_mm, y=ring_volts, mode="lines+markers",
                      name="ring DC (V)", yaxis="y2",
                      line=dict(color="#d62728", dash="dot"))
fig_taper.update_layout(
    title="the funnel, as declared: aperture taper and the DC ladder",
    xaxis_title="axial position x (mm)",
    yaxis_title="aperture radius (mm)",
    yaxis2=dict(title="ring DC (V)", overlaying="y", side="right"),
    width=FIGSIZE[0], height=FIGSIZE[1],
    margin=dict(l=70, r=70, t=52, b=52))
fig_taper

## Stage B — the solved field, and why ions do not hit the rings

The DC field alone would let an ion drift into a ring and stick. What
keeps it off the metal is the RF, through the **pseudopotential**: average
over the fast RF oscillation and the ion behaves as if it sat in a static
well,

**Eqn #3**

$$U_{\text{pseudo}}(\mathbf{r}) \;=\;
   \frac{e^2\,E_0(\mathbf{r})^2}{4\,m\,\Omega^2},$$

with $E_0$ the local RF field amplitude. **Why this equation matters
here:** it says the repulsion scales as $E_0^2$ — so it is enormous right
at the rings, where the RF field is strongest, and negligible on the axis.
That is a wall exactly where you want one, and nothing in the way where
you do not. It also scales as $1/m$: heavy ions are confined more weakly,
which is why a funnel has a low-mass and a high-mass limit.

The two panels below are the same solved device: the DC potential the
ladder makes, then the effective potential an m/z 556 ion actually sees.

In [ ]:
needs_solve = build_needs_solve(spec)
t0 = time.time()
model, fly, columns, births = build_run(spec)
print(f"build_run: {time.time() - t0:.2f} s "
      f"({'cold solve + cache store' if needs_solve else 'cache hit'})")
print(f"recorded channels: {columns}")

interactive_panel(scene_from_simspec(spec, model, field="phi"),
                  "xy", colorscale=COLORMAP, figsize=FIGSIZE)

In [ ]:
# ---- Parameters of the PE view -- change to suit your system ------------
PE_MZ = float(spec.source.mz_list[0])   # PE is ION-SPECIFIC: RF term ~ 1/m
PE_STRIDE = 2        # decimate the surface grid for a responsive 3-D render

# compute_component picks the component; pe_figure_3d renders the 
# style landscape. Passing the surface in lets both PE figures below share
# one reduction instead of recomputing it.
pe_surface = compute_component(model, PE_MZ, 1, "effective")
pe_landscape = pe_figure_3d(
    model=model, mz=PE_MZ, surface=pe_surface, stride=PE_STRIDE,
    height=FIGSIZE[1] + 140, show_adiabatic_caveat=True,
    title=f"{spec.name} · effective PE (DC + RF pseudopotential) · "
          f"m/z {PE_MZ:g}")
pe_landscape

Read it as a landscape: the ion is a ball rolling on this surface. The
**ridges at each ring** are the RF pseudopotential — they are what the ion
cannot climb, and they exist only because the RF is there. The **gentle
downhill slope along the axis** is the DC ladder, and it is what moves the
ion forward. The funnel works because those two are separable: a steep
wall where you want confinement, a shallow ramp where you want transport.

The `adiabatic approximation` note on the figure is not decoration. The
pseudopotential is an average over the RF cycle, and it is only meaningful
when the ion's motion is slow compared with that cycle. An ion that gains
enough energy to cross a ridge within one RF period is not described by
this picture at all — which is exactly the population that gets lost, and
exactly why the flown trajectories in the next stage are the check on
this figure rather than a decoration of it.

## Stage C — integration time versus record time

These are two different numbers and they answer two different questions.
Confusing them is one of the easiest ways to draw a wrong conclusion from
a perfectly correct simulation.

**`dt_ns` is the physics.** It is the step the integrator actually takes,
so it must resolve everything that pushes the ion. Here there are two such
timescales, and the smaller one wins:

**Eqn #4**

$$\Delta t \;\ll\; \min\!\left(T_{\text{RF}},\;
   \tau_{\text{collision}}\right),
\qquad T_{\text{RF}} = \frac{1}{f_{\text{RF}}},
\qquad \tau_{\text{collision}} \approx
   \frac{t_{\text{flight}}}{N_{\text{collisions}}}.$$

**Why the second term is easy to forget:** in vacuum only the RF matters,
and it is slow. Add a gas at 1 Torr and the ion is hit thousands of times
per flight; if `dt` is comparable to the mean free time, the integrator
steps over collisions and the damping is wrong — quietly, with no error
raised. The cell below measures both timescales from this run.

**`rec_every` is the recording.** It stores every *n*-th step and throws
the rest away. It changes **nothing** about the physics — the ion is
integrated at `dt` regardless — it changes only what you can see
afterwards and how much memory the trajectory costs:

**Eqn #5**

$$N_{\text{rows}} = \frac{t_{\text{max}}}{\Delta t \cdot
   \texttt{rec\_every}},
\qquad
N_{\text{samples per RF period}} = \frac{T_{\text{RF}}}
   {\Delta t \cdot \texttt{rec\_every}}.$$

The experiment below flies the *same ion* at three record intervals and
compares both the physics and what was recorded.

In [ ]:
# ---- Parameters of the decimation study -- change to suit your system ----
REC_EVERY_TRIALS = (8, 80, 400)     # record every n-th integrator step
DECIM_N_IONS     = 4                # ions per trial (ion 0 is compared)

rows = []
for rec_every in REC_EVERY_TRIALS:
    spec_rec = load_funnel(n_ions=DECIM_N_IONS, rec_every=int(rec_every))
    model_r, fly_r, columns_r, births_r = build_run(spec_rec)
    i_ke = columns_r.index("ke_ev")
    traj_r, summary_r = fly_r(0)          # same seed -> the SAME ion
    dt_us = spec_rec.integration.dt_ns * 1e-3
    rows.append({
        "rec_every": rec_every,
        "rows": len(traj_r),
        "sample_dt_ns": spec_rec.integration.dt_ns * rec_every,
        "samples_per_rf": T_RF_US / (dt_us * rec_every),
        "tof_us": summary_r["tof"],
        "x_end_mm": summary_r["x_end"],
        "ke_mean_ev": float(traj_r[:, i_ke].mean()),
        "ke_max_ev": float(traj_r[:, i_ke].max()),
        "megabytes": traj_r.nbytes / 1e6,
    })

header = ("| rec_every | rows | sample dt (ns) | samples/RF period | "
          "TOF (us) | x_end (mm) | KE mean (eV) | KE max (eV) | traj (MB) |")
table = [header, "|" + "---|" * 9]
for r in rows:
    table.append(
        f"| {r['rec_every']} | {r['rows']:,} | {r['sample_dt_ns']:.0f} | "
        f"{r['samples_per_rf']:.0f} | {r['tof_us']:.6f} | "
        f"{r['x_end_mm']:.6f} | {r['ke_mean_ev']:.4f} | "
        f"{r['ke_max_ev']:.4f} | {r['megabytes']:.2f} |")
display(Markdown("\n".join(table)))

tofs = {round(r["tof_us"], 9) for r in rows}
ends = {round(r["x_end_mm"], 9) for r in rows}
print(f"identical TOF across all record intervals: {len(tofs) == 1} "
      f"({', '.join(f'{t:.6f}' for t in sorted(tofs))} us)")
print(f"identical landing point:                   {len(ends) == 1}")
ke_max = [r["ke_max_ev"] for r in rows]
print(f"recorded KE maximum: {ke_max[0]:.3f} -> {ke_max[-1]:.3f} eV "
      f"({100*(1 - ke_max[-1]/ke_max[0]):.0f}% of the peak lost to "
      "decimation), while the mean moves by "
      f"{100*abs(rows[-1]['ke_mean_ev']/rows[0]['ke_mean_ev'] - 1):.1f}%")

Read the table twice.

**Down the physics columns** (TOF, landing point) nothing changes: the
integration is identical, exactly as it should be. If those columns *had*
moved, `rec_every` would be doing something it has no business doing.

**Down the recording columns** something is quietly lost. The mean kinetic
energy survives decimation — averages are robust to sampling — but the
recorded **maximum** falls, because the brief energetic excursions (an ion
freshly kicked by a collision, an ion at the top of its RF micromotion)
happen between the samples that were kept. A study of *heating*, of
fragmentation thresholds, or of anything living in the tail of a
distribution must record finely; a study of transmission, timing or
trajectory shape need not, and pays for the choice in memory.

**A practical rule:** choose `dt` from the physics, then choose
`rec_every` from the *smallest feature you intend to look at* — and state
both when quoting a result.

In [ ]:
# The two timescales that set dt, MEASURED from this run.
spec_meas = load_funnel(n_ions=8)
model_m, fly_m, columns_m, births_m = build_run(spec_meas)
summaries_m = [fly_m(i)[1] for i in range(len(births_m))]
mean_collisions = float(np.mean([s["n_col"] for s in summaries_m]))
mean_tof_us = float(np.mean([s["tof"] for s in summaries_m]))
tau_collision_ns = 1e3 * mean_tof_us / max(mean_collisions, 1.0)
dt_ns = float(spec_meas.integration.dt_ns)
print(f"RF period            : {T_RF_US*1e3:,.0f} ns "
      f"-> {T_RF_US*1e3/dt_ns:,.0f} integrator steps per RF cycle")
print(f"mean time between collisions: {tau_collision_ns:.1f} ns "
      f"-> {tau_collision_ns/dt_ns:.0f} integrator steps per collision")
print(f"({mean_collisions:,.0f} collisions per ion over {mean_tof_us:.0f} us "
      f"at {spec_meas.collisions.P_torr:.2f} Torr)")
print(f"\ndt = {dt_ns:g} ns is set by the COLLISIONS here, not the RF — "
      "the gas is the faster clock at this pressure.")

## Stage D — flying through the gas: what the funnel does to an ion

Now the flight itself. Watch two things at once: the ion is squeezed
**radially** toward the axis, and its **kinetic energy stays low** even
though a 30 V ladder is pulling it forward. Those two facts together are
the whole point of the device — it delivers ions from a high-pressure
region into a small aperture without heating them.

In [ ]:
spec_fly = load_funnel()
model_f, fly_f, columns_f, births_f = build_run(spec_fly)
i_x, i_y = columns_f.index("x"), columns_f.index("y")
i_ke = columns_f.index("ke_ev")

t0 = time.time()
trajectories, fates, summaries = [], [], []
for i in range(len(births_f)):
    traj_i, summary_i = fly_f(i)
    summaries.append(summary_i)
    if traj_i is not None and len(traj_i):
        trajectories.append(traj_i)
        fates.append(str(summary_i.get("kind", "")))
print(f"flew {len(trajectories)} ions in {time.time()-t0:.1f} s "
      f"({np.mean([s['n_col'] for s in summaries]):,.0f} collisions each)\n")
for line in describe_fates(spec_fly, model_f, summaries):
    print(line)

r_start = float(np.mean([abs(t[0, i_y]) for t in trajectories]))
r_end = float(np.mean([abs(t[-1, i_y]) for t in trajectories]))
print(f"\nradial compression: mean |r| {r_start:.2f} mm at the entrance -> "
      f"{r_end:.3f} mm at the exit ({r_start/max(r_end,1e-9):.0f}x)")
# What aperture would a real instrument need at the exit plate?
r_arrival = np.sort([abs(t[-1, i_y]) for t in trajectories])
for pct in (50, 95, 100):
    r_p = float(np.percentile(r_arrival, pct))
    print(f"  {pct:>3d}% of ions land within |r| = {r_p:.3f} mm "
          f"-> a {2*r_p:.2f} mm diameter conductance limit would pass them")

### The flight, on the instrument

Every flown ensemble in this notebook is now **shown on the instrument
it flew**: the solver's own electrode mask, the solved
potential, and the recorded paths — never a redrawing. The live panel
below it is for exploring; this figure is the record.

In [ ]:
# The flown ensemble rendered on the solver geometry, through the
# framework (viz_core), sized to one screen and backend-proof.
from ion_gym.viz.viz_core import scene_from_simspec, render_mpl
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt
sc_fly = scene_from_simspec(
    spec_fly, model_f, field="phi", trajs=trajectories, fates=fates,
    title=(f"Ion funnel, Stage D flight — {len(trajectories)} ions, "
           f"m/z {spec_fly.source.mz_list} | RF "
           f"{spec_fly.geometry.rf_groups[0].amplitude_v:g} V @ "
           f"{spec_fly.geometry.rf_groups[0].frequency_hz/1e3:g} kHz | "
           f"{spec_fly.collisions.gas} {spec_fly.collisions.P_torr:g} Torr"))
fig = render_mpl(sc_fly, views=["xy"])
_buf = _io.BytesIO()
fig.savefig(_buf, format="png", dpi=100, bbox_inches="tight")
_display(_PNG(_buf.getvalue(), height=470))
_plt.close(fig)


In [ ]:
interactive_panel(
    scene_from_simspec(spec_fly, model_f, field="phi",
                       trajs=trajectories, fates=fates),
    "xy", colorscale=COLORMAP, figsize=FIGSIZE)

### The same landscape, with the ions on it

The figure above was the landscape alone. This is the landscape with the
flown trajectories draped on it — the ions are drawn at the PE of the
point they occupy. This is the check on the previous figure: if the
pseudopotential picture is right, ions should ride the valley floor and
turn back from the ridges.

**Path colour** is a parameter. By default each path is coloured by its
**fate**, using the same map as every other figure in the toolkit
(splat, boundary exit, timeout, declared plane) — which is informative
when an ensemble ends several different ways. In *this* device every ion
ends the same way, so the fate palette says nothing and a single colour
reads better; set `TRAJ_COLOR` below to a colour string for that, or to a
`{fate: colour}` dict to highlight one outcome against the rest.

In [ ]:
# ---- Parameters of the PE overlay -- change to suit your system ---------
MAX_TRAJ_ON_PE = 8     # ion paths draped on the landscape
PE_DECIMATE = 4        # sample every n-th recorded point when draping
TRAJ_COLOR = None      # None = colour each path by its FATE (the default);
                       # "#00e5ff" = one colour for all paths;
                       # {0: "#ffffff", 2: "#00e5ff"} = per-fate override
TRAJ_WIDTH = 3         # line width of the draped paths

# pe_figure_3d takes the SAME record type the ensemble driver produces,
# so the overlay is the flight itself rather than a re-run.
ion_records = [IonResult(index=i, traj=trajectories[i],
                         summary=summaries[i])
               for i in range(len(trajectories))]
pe_with_ions = pe_figure_3d(
    model=model_f, mz=PE_MZ, surface=pe_surface, stride=PE_STRIDE,
    results=ion_records, max_traj=MAX_TRAJ_ON_PE, decimate=PE_DECIMATE,
    traj_color=TRAJ_COLOR, traj_width=TRAJ_WIDTH,
    height=FIGSIZE[1] + 140, show_adiabatic_caveat=True,
    title=f"{spec.name} · effective PE with {MAX_TRAJ_ON_PE} flown ions "
          f"· m/z {PE_MZ:g}")
pe_with_ions

### The requested plot: kinetic energy versus distance

Each thin line is one ion; the heavy line is the ensemble mean in axial
bins, and the shaded band is the 5th-95th percentile — the *spread* is as
informative as the mean, because it is the tail that fragments molecules.

What to look for:

* The mean sits at a few tenths of an eV for most of the device, even
  though the ladder is pulling the ion through a 30 V drop. Every eV the
  field delivers is handed to the gas within a few collisions.
* The band is asymmetric: brief upward excursions, no downward ones. That
  is the signature of a collisional steady state — the field pumps energy
  in continuously, collisions remove it in discrete kicks.
* Near the exit the mean climbs. The aperture is smallest there, the axial
  field is strongest, and the ion begins to outrun the damping — which is
  exactly where a real instrument places its next pumping stage.

In [ ]:
# ---- Parameters of the KE profile -- change to suit your system ----------
N_AXIAL_BINS = 40
N_TRACES_SHOWN = 12     # individual ions drawn behind the statistics

x_all = np.concatenate([t[:, i_x] for t in trajectories])
ke_all = np.concatenate([t[:, i_ke] for t in trajectories])
edges = np.linspace(float(x_all.min()), float(x_all.max()), N_AXIAL_BINS + 1)
which = np.digitize(x_all, edges)
centres, ke_mean, ke_lo, ke_hi = [], [], [], []
for k in range(1, len(edges)):
    sel = ke_all[which == k]
    if sel.size < 20:            # a bin too sparse to summarise is SKIPPED,
        continue                 # visibly, rather than plotted as noise
    centres.append(0.5 * (edges[k-1] + edges[k]))
    ke_mean.append(float(sel.mean()))
    ke_lo.append(float(np.percentile(sel, 5)))
    ke_hi.append(float(np.percentile(sel, 95)))
print(f"{len(centres)} of {N_AXIAL_BINS} axial bins had enough samples "
      "to summarise; the rest are omitted rather than drawn as noise.")

fig_ke = go.Figure()
for traj_i in trajectories[:N_TRACES_SHOWN]:
    fig_ke.add_scatter(x=traj_i[:, i_x], y=traj_i[:, i_ke], mode="lines",
                       line=dict(width=0.7), opacity=0.35,
                       showlegend=False, hoverinfo="skip")
fig_ke.add_scatter(x=centres + centres[::-1], y=ke_hi + ke_lo[::-1],
                   fill="toself", fillcolor="rgba(0,229,255,0.18)",
                   line=dict(width=0), name="5th-95th percentile",
                   hoverinfo="skip")
fig_ke.add_scatter(x=centres, y=ke_mean, mode="lines",
                   line=dict(color="#00e5ff", width=3),
                   name="ensemble mean")
fig_ke.update_layout(
    title=f"kinetic energy vs distance through the funnel "
          f"(m/z {spec_fly.source.mz_list[0]:g}, "
          f"{spec_fly.collisions.gas} at {spec_fly.collisions.P_torr:.2f} "
          f"Torr, {N_IONS} ions)",
    xaxis_title="axial position x (mm)",
    yaxis_title="kinetic energy (eV)",
    width=FIGSIZE[0], height=FIGSIZE[1],
    margin=dict(l=70, r=20, t=52, b=52))
fig_ke

### The energy budget

The statement above — "every eV the field delivers is handed to the gas" —
is checkable, so check it. The field's work on an ion crossing the device
is the drop in the on-axis potential between entrance and exit, read from
the **solved** field:

**Eqn #6**

$$W_{\text{field}} \;=\; e\,\bigl[\Phi(x_{\text{in}})
   - \Phi(x_{\text{out}})\bigr],
\qquad
\frac{\langle \mathrm{KE} \rangle}{W_{\text{field}}}
   \;=\; \text{the fraction the ion keeps.}$$

In [ ]:
phi_grid = np.asarray(model_f.A, float)
h_mm = float(getattr(model_f, "mm_per_gu", None) or model_f.h_mm)
phi_axis = phi_grid[:, 0]                       # r = 0 row: the beam axis
x_axis_mm = np.arange(len(phi_axis)) * h_mm

x_in, x_out = float(x_all.min()), float(x_all.max())
phi_in = float(np.interp(x_in, x_axis_mm, phi_axis))
phi_out = float(np.interp(x_out, x_axis_mm, phi_axis))
work_ev = phi_in - phi_out
ke_mean_flight = float(np.mean([t[:, i_ke].mean() for t in trajectories]))
print(f"on-axis potential: {phi_in:.1f} V at x = {x_in:.2f} mm -> "
      f"{phi_out:.1f} V at x = {x_out:.2f} mm")
print(f"field work per ion         : {work_ev:.1f} eV")
print(f"mean kinetic energy carried: {ke_mean_flight:.3f} eV")
print(f"-> the ion keeps {100*ke_mean_flight/work_ev:.1f}% of the work; "
      f"{100*(1 - ke_mean_flight/work_ev):.1f}% goes into the gas")
print(f"   ({np.mean([s['n_col'] for s in summaries]):,.0f} collisions per "
      f"ion, i.e. ~{1e3*work_ev/np.mean([s['n_col'] for s in summaries]):.2f} "
      "meV handed over per collision on average)")

### The control: the same device with the gas switched off

One line in the spec turns the gas off. Nothing else changes — same
geometry, same fields, same ions, same seed — so whatever differs is the
gas and only the gas.

In [ ]:
spec_vacuum = load_funnel(gas_enabled=False)   # the only change
model_v, fly_v, columns_v, births_v = build_run(spec_vacuum)
i_ke_v = columns_v.index("ke_ev")
i_y_v = columns_v.index("y")

trajectories_v, summaries_v = [], []
for i in range(len(births_v)):
    traj_i, summary_i = fly_v(i)
    summaries_v.append(summary_i)
    if traj_i is not None and len(traj_i):
        trajectories_v.append(traj_i)

def exit_ke(trajs, ike):
    return float(np.mean([t[-1, ike] for t in trajs]))

ke_gas = exit_ke(trajectories, i_ke)
ke_vac = exit_ke(trajectories_v, i_ke_v)
r_gas = float(np.mean([abs(t[-1, i_y]) for t in trajectories]))
r_vac = float(np.mean([abs(t[-1, i_y_v]) for t in trajectories_v]))
print(f"{'':>12s}  {'exit KE (eV)':>13s}  {'exit |r| (mm)':>14s}  "
      f"{'ions returned':>14s}")
print(f"{'with gas':>12s}  {ke_gas:13.3f}  {r_gas:14.3f}  "
      f"{len(trajectories):>14d}")
print(f"{'vacuum':>12s}  {ke_vac:13.3f}  {r_vac:14.3f}  "
      f"{len(trajectories_v):>14d}")
if ke_vac > ke_gas:
    print(f"\n-> without the gas the ion arrives {ke_vac/max(ke_gas,1e-9):.0f}x "
          "hotter: the ladder's work now has nowhere to go. The funnel is "
          "not a lens that happens to sit in a gas — the gas is the "
          "component that makes it work.")
else:
    print(f"\n-> at this operating point the vacuum flight is NOT hotter "
          f"({ke_vac:.3f} vs {ke_gas:.3f} eV); read the fates above before "
          "drawing a conclusion — ions lost early never sample the ladder.")

____

## Where the series goes next

* **SLIM**: travelling-wave drives, where the confinement is RF but the
  transport is a moving DC wave rather than a static ladder — and the
  record-versus-integrate question returns, because the wave adds a third
  timescale.

The decimation study in Stage C is worth repeating on any new device: it
costs three flights and it tells you, quantitatively, what your stored
trajectories can and cannot answer.

## Read-out — what this notebook established

- **Gas changes the problem qualitatively.** Without collisions an ion's energy is set by the fields alone; with a buffer gas the ion reaches a drift equilibrium in which the RF supplies confinement and the DC gradient supplies transport, and the effective temperature is set by the competition between them.
- **The funnel focuses by shrinking the confining aperture along the axis** while the DC ladder keeps pushing — so the transmitted beam is both narrower and, in the RF field it now samples, hotter. The trade between transmission and effective temperature is the design tension of every funnel.
- **What to watch in the numbers:** transmission alone can mislead. A funnel that passes everything into a hot, spread-out beam has moved the problem downstream rather than solving it — quote transmission and effective temperature together, at a stated pressure and RF amplitude.